# Candidate Generation

In [2]:

import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\yeshw\Documents\Gayathri personal doc\Gayathri\arizona\INFO 556 - Text Retrieval and Web Search\project\hm-fashion-recommendation


In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
import pandas as pd
from config import ROOT_DIR
from src.models.baseline_model import create_baseline_recommender,create_recency_baseline
from src.models.als_model import train_and_save_als_model, create_als_recommender
from src.evaluation.metrics import evaluate_all, evaluate_candidates
from src.candidate_retrieval.PreviousPurchases import PreviousPurchases
from src.candidate_retrieval.ItemSimilarity import ItemSimilarityCandidates
from src.candidate_retrieval.RecencyCandidates import RecencyCandidates
from src.candidate_retrieval.AlsCandidates import ALSCandidates
from src.candidate_retrieval.StyleAlternatives import SameProductColorCandidates
from src.candidate_retrieval.CandidateRetrievalPipeline import CandidateGeneratorEnsemble

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
cleaned_df_folder = "input/cleaned"
def load_from_parquet(cleaned_df_name):
    print("Loading data from Parquet...")
    return pd.read_parquet(cleaned_df_folder + "/" + cleaned_df_name + ".parquet")

In [6]:
cols_needed = ['t_dat', 'customer_id', 'article_id','week']

train_transactions = pd.read_parquet(
    "input/cleaned/train_transactions.parquet", 
    columns=cols_needed
)

In [7]:

articles = load_from_parquet("articles")
customers = load_from_parquet("customers")

Loading data from Parquet...
Loading data from Parquet...


In [ ]:
# week 2 is the latest week in the training data, so we will use it as the target week for training our model. 
# The transactions from week 2 will be used to create the labels for our training data, while the transactions from weeks 3 to 105 will be used as the historical data for generating candidates.
train_target_week = 2
train_target_tx = train_transactions[train_transactions['week'] == train_target_week] # week 2
train_history_tx = train_transactions[train_transactions['week'] > train_target_week] # Weeks 3 to 105

### Baseline model

In [11]:
# Initialize the baseline recommender to get the top-k recommendations.
my_baseline_model = create_baseline_recommender(train_history_tx)

# Evaluate the baseline model on the validation transactions.
evaluate_all(my_baseline_model, train_target_tx)

Generating predictions for 72,019 validation customers...
Preparing evaluation matrices...
Computing metrics...


{'MAP@12': 0.0034440087457057516,
 'HitRate@12': 0.02477124092253433,
 'NDCG@12': 0.006318058900555214,
 'Recall@12': 0.008970650289728576}

### Recency model

In [12]:
# Evaluate the recency baseline model on the training target transactions.
recency_model = create_recency_baseline(train_history_tx)
evaluate_all(recency_model, train_target_tx)


Generating predictions for 72,019 validation customers...
Preparing evaluation matrices...
Computing metrics...


{'MAP@12': 0.005770575041379203,
 'HitRate@12': 0.055485358030519724,
 'NDCG@12': 0.012033335990458138,
 'Recall@12': 0.02042971796233196}

### Collaborative filtering using ALS modelling

The training dataset is massive with around 1.3M unique customers and around 100k unique articles. This makes the collaborative filtering model to run very slowly on the entire dataset as the sparse matrix is massive.

So, down sampling only the last 2 months records would help in running the ALS model faster and also the data would be more relative to recommend as the customer's fashion sense from last 2 years would not matter much.

In [ ]:

als_train_history_file_path = ROOT_DIR / 'saved_models' / 'als_recommender_model_training_history.pkl'
# Train als model for train_history
train_and_save_als_model(
    train=train_history_tx,
    model_path=als_train_history_file_path,
    lookback_weeks=14,
    factors=100,
    iterations=15,
    regularization=0.01,
    alpha=40.0,
    random_seed=42
)

In [ ]:
# Load the ALS model and mappings 

with open(als_train_history_file_path, "rb") as f:
    artifacts = pickle.load(f)
    
als_recommender = create_als_recommender(
    als_model=artifacts["model"],
    user_map=artifacts["user_map"],
    item_map=artifacts["item_map"],
    reverse_item_map=artifacts["reverse_item_map"],
    user_item_matrix=artifacts["user_item_matrix"],
    baseline=recency_model  # Fallback for cold-start or unknown customers
)
evaluate_all(als_recommender, train_target_tx)

### Candidate Selection


#### Previous Purchases

In [13]:
train_target_customers = train_target_tx['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 20}),
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_history_tx, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(train_target_customers):,} validation"
    " customers..."
)
prev_recs_df = ensemble.generate(users=train_target_customers)

print(f"Generated {len(prev_recs_df):,} total candidate rows.")
prev_recs_df.head()

Generating candidates via ensemble for 72,019 validation customers...
Generated previous purchase candidates
Generated 420,413 total candidate rows.


,customer_id,article_id,source
420412,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0570004009,previous_purchases
420411,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0816759002,previous_purchases
420410,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0795975009,previous_purchases
420409,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0835561003,previous_purchases
420408,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0636323002,previous_purchases


In [15]:
evaluate_candidates(prev_recs_df, train_target_tx, k=20)

Preparing evaluation matrices...
Recall@20: 0.0362


0.0361941828449441

#### Item Similarity

In [9]:

train_target_customers = train_target_tx['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 20}),
    "item_cooccurrence": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "k": 30
        },
    ),
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_history_tx, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(train_target_customers):,} validation"
    " customers..."
)
sim_recs_df = ensemble.generate(users=train_target_customers)

print(f"Generated {len(sim_recs_df):,} total candidate rows.")
evaluate_candidates(sim_recs_df, train_target_tx, k=50)

Generating candidates via ensemble for 72,019 validation customers...
Generated previous purchase candidates
Generated candidate scores for users
Generated item similarity candidates
Generated 1,251,056 total candidate rows.
Preparing evaluation matrices...
Recall@50: 0.0624


0.06238366851558556

#### Adding Recency to the ensemble

In [ ]:
train_target_customers = train_target_tx['customer_id'].unique()
generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 20}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "k": 30
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_history_tx, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(train_target_customers):,} validation"
    " customers..."
)
recs_df_3 = ensemble.generate(users=train_target_customers)

print(f"Generated {len(recs_df_3):,} total candidate rows.")
evaluate_candidates(recs_df_3, train_target_tx, k=70)


Generating candidates via ensemble for 72,019 validation customers...
Generated previous purchase candidates
Generated candidate scores for users
Generated item similarity candidates
Generated recency candidates
Generated 2,766,872 total candidate rows.
Preparing evaluation matrices...
Recall@70: 0.0908


0.09084169024721206

#### ALS model

In [20]:
als_train_history_file_path = PROJECT_ROOT / 'saved_models' / 'als_recommender_model_training_history.pkl'

In [ ]:
# Train als model for train_history
train_and_save_als_model(
    train=train_history_tx,
    model_path=als_train_history_file_path,
    lookback_weeks=14,
    factors=100,
    iterations=15,
    regularization=0.01,
    alpha=40.0,
    random_seed=42
)

In [24]:
train_target_customers = train_target_tx['customer_id'].unique()

generators_config = {
    "als": (ALSCandidates, {"model_path": als_train_history_file_path, "k": 40})
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_history_tx, articles=articles, generators=generators_config
)

# Generate candidates for train target customers
print(
    f"Generating candidates via ensemble for {len(train_target_customers):,} train target"
    " customers..."
)
als_recs_df = ensemble.generate(users=train_target_customers)

print(f"Generated {len(als_recs_df):,} total candidate rows.")
evaluate_candidates(als_recs_df, train_target_tx, k=100)

Generating candidates via ensemble for 72,019 train target customers...
Generated als candidates
Generated 1,791,520 total candidate rows.
Preparing evaluation matrices...
Recall@100: 0.0661


0.06614797606783407

In [ ]:
train_target_customers = train_target_tx['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 15}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "k": 25
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
    "als": (ALSCandidates, {"model_path": als_train_history_file_path, "k": 40})
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_history_tx, articles=articles, generators=generators_config
)

# Generate candidates for train target customers
print(
    f"Generating candidates via ensemble for {len(train_target_customers):,} train target"
    " customers..."
)
recs_df_1 = ensemble.generate(users=train_target_customers)

print(f"Generated {len(recs_df_1):,} total candidate rows.")
evaluate_candidates(recs_df_1, train_target_tx, k=100)

Generating candidates via ensemble for 72,019 train target customers...
Generated previous purchase candidates
Generated candidate scores for users
Generated item similarity candidates
Generated recency candidates
Generated als candidates
Generated 3,833,215 total candidate rows.
Preparing evaluation matrices...
Recall@100: 0.1090


0.1090016095857673

#### Adding Style Alternative to the ensemble

In [22]:

train_target_customers = train_target_tx['customer_id'].unique()

generators_config = {
    "same_product_color": (SameProductColorCandidates, {"lookback_weeks": 12, "k": 20})
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_history_tx, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(train_target_customers):,} validation"
    " customers..."
)
style_recs_df = ensemble.generate(users=train_target_customers)

print(f"Generated {len(style_recs_df):,} total candidate rows.")
evaluate_candidates(style_recs_df, train_target_tx, k=20)
style_recs_df.head()

Generating candidates via ensemble for 72,019 validation customers...
Generated style alternatives candidates
Generated 1,013,034 total candidate rows.
Preparing evaluation matrices...
Recall@20: 0.0044


,customer_id,article_id,source
369442,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0699924001,same_product_color
369432,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0658329031,same_product_color
369446,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0395730041,same_product_color
369445,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0733090001,same_product_color
369444,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0791498009,same_product_color


In [ ]:
train_target_customers = train_target_tx['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 13, "k": 15}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 7,
            "min_item_frequency": 30,
            "k": 25
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 13, "k": 20}),
    "als": (ALSCandidates, {"model_path": als_train_history_file_path, "k": 40}),
    "same_product_color": (SameProductColorCandidates, {"lookback_weeks": 13, "k": 20})
}
ensemble = CandidateGeneratorEnsemble(
    transactions=train_history_tx, articles=articles, generators=generators_config
)
# Generate candidates for train target customers
print(
    f"Generating candidates via ensemble for {len(train_target_customers):,} train target"
    " customers..."
)
train_target_recs_df = ensemble.generate(users=train_target_customers)

print(f"Generated {len(train_target_recs_df):,} total training candidate rows.")
evaluate_candidates(train_target_recs_df, train_target_tx, k=120)

Generating candidates via ensemble for 72,019 train target customers...
Generated previous purchase candidates
Generated candidate scores for users
Generated item similarity candidates
Generated recency candidates
Generated als candidates
Generated style alternatives candidates
Generated 4,939,981 total training candidate rows.
Preparing evaluation matrices...
Recall@120: 0.1124


0.112412707215924